# Head-Only Detector — Hybrid Architecture Companion

Trains a YOLOv8n single-class **head** detector on the *OverHead Head
Detection* dataset (top-down camera angle — matches the cabin CCTV
scenario). The resulting `best_head.pt` is plugged into
`scripts/run_simulation.py` via `--head-weights` for the hybrid
two-model occupancy estimator described in the thesis methodology.

## Why a separate head model?

Single-frame elevator CCTV exhibits two systematic failure modes for
the four-class detector:

1. **Body occlusion in crowded cabins** — closely packed passengers
   share a visual silhouette so the detector merges or drops boxes.
2. **Top-down class confusion** — overhead poses look geometrically
   similar to luggage / boxes.

A head-only detector is robust to both: heads are always at the top
of every cabin occupant and remain visible regardless of crowding.

## Inputs expected on Drive

Place these in `MyDrive/Capstone/`:

| File | Source |
|---|---|
| `overhead_head.zip` | Rename `OverHead Head Detection.yolov8.zip` to `overhead_head.zip` and upload |
| `05_head_model_training.ipynb` | This notebook |

## Outputs written to Drive

* `MyDrive/Capstone/models/runs/elevator_head_v1/` — training run
  artifacts (loss curves, PR / F1 plots, confusion matrix)
* `MyDrive/Capstone/models/weights/best_head.pt` — production checkpoint

## 1. Mount Drive and prepare workspace

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, zipfile
from pathlib import Path

BASE_DRIVE = '/content/drive/MyDrive/Capstone'
WORK = '/content/work'
DATA_DIR = f'{WORK}/data/overhead_head'

ZIP_PATH = f'{BASE_DRIVE}/overhead_head.zip'
assert os.path.exists(ZIP_PATH), (
    f'overhead_head.zip not found at {ZIP_PATH}. Upload it to MyDrive/Capstone/.'
)

os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(DATA_DIR)
print('Extracted to:', DATA_DIR)
%cd {DATA_DIR}
!ls

## 2. Install dependencies

In [ ]:
!pip install -q ultralytics

## 3. Consolidate labels — collapse all classes to a single `head` class

The downloaded dataset has four classes
(`head`, `head-top-view`, `people`, `person`). Our hybrid pipeline
only needs *one* head class, so we rewrite every label file to use
class id `0` and overwrite `data.yaml` with a single-class spec.

This keeps the bounding boxes intact (they all describe heads or
head-shoulder regions) while turning the problem into a single-class
detection task — easier to train and easier to ensemble at inference.

In [ ]:
from pathlib import Path
import yaml

data_yaml_path = Path(DATA_DIR) / 'data.yaml'
with open(data_yaml_path) as f:
    cfg = yaml.safe_load(f)
print('Original classes:', cfg.get('names'))

# Auto-detect existing split directories. The Roboflow ZIP usually
# contains 'train', 'valid', and 'test', but some exports use 'val'
# instead of 'valid' or omit a split entirely.
DATA_ROOT = Path(DATA_DIR)
existing_splits: dict[str, Path] = {}
for cand in ('train', 'valid', 'val', 'test'):
    if (DATA_ROOT / cand / 'images').is_dir():
        existing_splits[cand] = DATA_ROOT / cand
print('Detected split directories:', list(existing_splits.keys()))

# Rewrite every label so the class id is always 0.
rewritten = 0
for split, split_dir in existing_splits.items():
    lbl_dir = split_dir / 'labels'
    if not lbl_dir.is_dir():
        continue
    for lbl in lbl_dir.glob('*.txt'):
        new_lines = []
        with open(lbl) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                new_lines.append(' '.join(['0'] + parts[1:]))
        with open(lbl, 'w') as f:
            f.write('\n'.join(new_lines))
        rewritten += 1
print(f'Rewrote {rewritten} label files to single class.')

# Resolve which directory to use for each YOLO split.
# YOLO requires train + val. Test is optional.
if 'train' not in existing_splits:
    raise RuntimeError(
        f'No train/ split found in {DATA_ROOT}. The dataset zip may not '
        'have extracted correctly.'
    )

if 'valid' in existing_splits:
    val_rel = 'valid/images'
elif 'val' in existing_splits:
    val_rel = 'val/images'
elif 'test' in existing_splits:
    # Fall back to test/ as validation (uncommon but recoverable).
    val_rel = 'test/images'
    print('[warn] no valid/ or val/ split — falling back to test/ for validation.')
else:
    # No validation split at all — split train 80/20 on the fly by symlink.
    import random as _random

    _random.seed(42)
    train_imgs = sorted((DATA_ROOT / 'train' / 'images').glob('*'))
    n_val = max(1, len(train_imgs) // 5)
    val_imgs = train_imgs[:n_val]
    val_dir = DATA_ROOT / 'valid'
    (val_dir / 'images').mkdir(parents=True, exist_ok=True)
    (val_dir / 'labels').mkdir(parents=True, exist_ok=True)
    for img in val_imgs:
        (val_dir / 'images' / img.name).symlink_to(img)
        lbl_src = DATA_ROOT / 'train' / 'labels' / (img.stem + '.txt')
        if lbl_src.exists():
            (val_dir / 'labels' / lbl_src.name).symlink_to(lbl_src)
    print(f'[info] created synthetic valid/ from {n_val} train images.')
    val_rel = 'valid/images'

test_rel = None
for cand in ('test', 'valid', 'val'):
    if cand in existing_splits and f'{cand}/images' != val_rel:
        test_rel = f'{cand}/images'
        break
if test_rel is None:
    test_rel = val_rel  # fallback so cell 12 (test eval) doesn't crash

# Overwrite data.yaml with a single-class spec and the resolved paths.
cfg['names'] = ['head']
cfg['nc'] = 1
cfg['path'] = str(DATA_ROOT)
cfg['train'] = 'train/images'
cfg['val'] = val_rel
cfg['test'] = test_rel
with open(data_yaml_path, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print('\nNew data.yaml:')
print(open(data_yaml_path).read())

## 4. GPU + dataset audit

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
!nvidia-smi 2>/dev/null | head -10

for split in ('train', 'valid', 'test'):
    img_dir = Path(DATA_DIR) / split / 'images'
    if img_dir.is_dir():
        n = sum(1 for _ in img_dir.iterdir())
        print(f'  {split:<6}: {n} images')

## 5. Train (max-quality configuration)

Hyperparameters tuned for an L4 GPU and the ~6k single-class dataset,
prioritising detection quality over training speed (overnight run is
fine). Expected wall-time: **~6 – 7 hours** for 150 epochs (early
stopping typically fires around epoch 80–110).

### Key choices

| Hyperparameter | Value | Justification |
|---|:---:|---|
| `model` | `yolov8s.pt` | Larger backbone than nano — modest accuracy gain at the cost of ~2× FLOPs (still real-time) |
| `imgsz` | **1280** | Doubled from 640: heads in elevator CCTV are often small (≈ 30–50 px), and higher resolution measurably improves recall on small objects |
| `batch` | **8** | Halved from the default to keep GPU usage well under the 22 GB L4 budget at `imgsz=1280` |
| `epochs` | 150 | Early stopping (`patience=30`) handles convergence; the higher cap covers slower mAP saturation at large `imgsz` |
| `multi_scale` | **False** | Disabled deliberately — random resize beyond 1280 (up to 1.5×) caused OOM events and silent CPU fallback. Mosaic + mixup + erasing already provide ample scale variability |
| `cos_lr` | True | Cosine schedule produces smoother convergence on long runs |
| `lr0` / warmup | 0.01 / 3 epochs | YOLOv8 defaults — already tuned |

### Augmentation policy (made explicit for thesis transparency)

| Hyperparameter | Value | Justification |
|---|:---:|---|
| `hsv_h` / `hsv_s` / `hsv_v` | 0.015 / 0.7 / 0.4 | Light-temperature variability of cabin LED / fluorescent fixtures |
| `degrees` | 10 | CCTV mounts can sit at slight angles; train rotation invariance |
| `translate` | 0.10 | Heads can land anywhere in frame |
| `scale` | 0.50 | Distance variability — heads near vs. far from the fish-eye lens |
| `fliplr` | 0.5 | Cabin geometry is left/right symmetric |
| `flipud` | 0.0 | Heads are always above shoulders — vertical flip would be unrealistic |
| `mosaic` | 1.0 | 4-image composites — synthetic crowd density |
| `mixup` | 0.10 | Light blending between samples — soft regularisation |
| `erasing` | 0.40 | Random rectangular occlusion — simulates occluded heads in dense crowds |
| `auto_augment` | `randaugment` | Selects a random sub-policy each batch (Cubuk et al. 2020) |

These yield roughly **30–40× effective dataset diversity** on the fly,
which is why we do not also pre-augment the head dataset offline.

In [ ]:
from ultralytics import YOLO

# Max-quality, OOM-safe config — see preceding markdown.
VARIANT = 'yolov8s.pt'   # nano -> small
EPOCHS  = 150            # early stopping handles overrun
BATCH   = 8              # 16 -> 8 (1280 imgsz needs the headroom on a 22 GB L4)
IMGSZ   = 1280           # 640 -> 1280 (small-object recall boost)

RUNS_DIR = f'{BASE_DRIVE}/models/runs'
os.makedirs(RUNS_DIR, exist_ok=True)

model = YOLO(VARIANT)
results = model.train(
    data=str(data_yaml_path),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    lr0=0.01,
    cos_lr=True,           # cosine learning-rate schedule
    multi_scale=False,     # off: random resize up to 1.5× imgsz blew past memory
    patience=30,
    seed=42,
    project=RUNS_DIR,
    name='elevator_head_v1',
    plots=True,
    save_period=10,
    device=0 if torch.cuda.is_available() else 'cpu',

    # === Explicit augmentation policy ===
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.10,
    scale=0.50,
    shear=0.0,
    perspective=0.0,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
    mixup=0.10,
    erasing=0.40,
    auto_augment='randaugment',
)
BEST_HEAD = f'{results.save_dir}/weights/best.pt'
print('best head checkpoint:', BEST_HEAD)

## 6. Evaluate on the held-out test split

In [ ]:
model_head = YOLO(BEST_HEAD)
metrics = model_head.val(data=str(data_yaml_path), split='test')

# Comprehensive metrics for the thesis Results chapter.
print(f"Precision (mP):   {metrics.box.mp:.4f}")
print(f"Recall (mR):      {metrics.box.mr:.4f}")
print(f"mAP@50:           {metrics.box.map50:.4f}")
print(f"mAP@50-95:        {metrics.box.map:.4f}")
f1 = 2 * metrics.box.mp * metrics.box.mr / (metrics.box.mp + metrics.box.mr + 1e-9)
print(f"F1 (derived):     {f1:.4f}")

# Per-class breakdown (single class here, but kept for clean tabular format).
print("\nPer-class detail:")
for i, name in metrics.names.items():
    print(
        f"  {name:<10} P={metrics.box.p[i]:.3f}  R={metrics.box.r[i]:.3f}  "
        f"mAP50={metrics.box.ap50[i]:.3f}  mAP50-95={metrics.box.ap[i]:.3f}"
    )

# Persist metrics to Drive next to the checkpoint for easy citation later.
import json

metrics_dict = {
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "map50": float(metrics.box.map50),
    "map50_95": float(metrics.box.map),
    "f1": float(f1),
}
metrics_path = f'{BASE_DRIVE}/models/weights/best_head_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics_dict, f, indent=2)
print(f"\nMetrics saved to: {metrics_path}")

## 7. Save the head checkpoint to Drive

`best_head.pt` lives next to `best.pt` (the four-class checkpoint).
The hybrid simulation script picks them up via `--weights` and
`--head-weights`.

In [ ]:
import shutil
drive_dst = f'{BASE_DRIVE}/models/weights/best_head.pt'
os.makedirs(os.path.dirname(drive_dst), exist_ok=True)
shutil.copy2(BEST_HEAD, drive_dst)
print('copied:', drive_dst, f'({os.path.getsize(drive_dst) / 1e6:.1f} MB)')

## 8. Smoke test on a couple of cabin frames

Optional. Drop one or two cabin photos into `/content/work/probe/` and
re-run the cell to visualize how many heads the new model finds.

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

probe_dir = f'{WORK}/probe'
os.makedirs(probe_dir, exist_ok=True)
probe_imgs = sorted(
    glob.glob(f'{probe_dir}/*.jpg') + glob.glob(f'{probe_dir}/*.png')
)[:6]
if not probe_imgs:
    print(f'Drop a few sample images at {probe_dir}/ then re-run this cell.')
else:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, p in zip(axes.flat, probe_imgs):
        res = model_head.predict(p, conf=0.25, verbose=False)[0]
        n_heads = 0 if res.boxes is None else len(res.boxes)
        ax.imshow(Image.fromarray(res.plot()[..., ::-1]))
        ax.set_title(f'{os.path.basename(p)} — {n_heads} heads', fontsize=9)
        ax.axis('off')
    plt.tight_layout(); plt.show()

## 9. What to do back on the local machine

1. Drive will sync `best_head.pt` to your local Capstone folder.
   Otherwise download it manually from
   `MyDrive/Capstone/models/weights/best_head.pt` and place it under
   `models/weights/best_head.pt` in the project.
2. Re-run the energy simulation in **hybrid mode**:

   ```
   python -m scripts.run_simulation \
       --images data/sim/images \
       --ground-truth data/sim/ground_truth.csv \
       --weights models/weights/best.pt \
       --head-weights models/weights/best_head.pt \
       --rated-capacity 8 \
       --num-calls 1000 \
       --output results/simulation/hybrid
   ```

3. Compare `results/simulation/baseline/` and
   `results/simulation/hybrid/` to populate the ablation table for the
   thesis Results chapter.